# Verify Threshold Tables and Confusion Matrix Figures

Use this notebook to make sure your LWCNN/CNN v3 and CNN-Transformer figures, threshold sweep tables, and report values all come from the **same predictions**.

The notebook will:

1. Load internal and external prediction CSVs for each model.
2. Compute confusion matrices at selected thresholds.
3. Compute the full threshold sweep table.
4. Plot the threshold-metric figure.
5. Check whether the reported values match the generated figures.
6. Save updated tables and figures for your report.

Important: do **not** mix a threshold table from one run with confusion matrices from another run. This notebook prevents that by calculating everything from the same prediction files.

## 1. Setup

Run this first. No training is performed in this notebook.

In [ ]:
import os
import glob
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Find prediction/result files in Drive

Run this cell to search for likely CSV/NPY/NPZ prediction files.  
You are looking for files that contain true labels and predicted probabilities, for example columns like:

- `y_true`
- `label`
- `true_label`
- `prob`
- `probability`
- `y_prob`
- `drone_probability`
- `clip_id` if the file is window-level and needs clip-level aggregation

If you cannot find prediction files, rerun your evaluation notebooks and save the prediction arrays to CSV.

In [3]:
SEARCH_ROOT = "/content/drive/MyDrive"

patterns = [
    "**/*.csv",
    "**/*.parquet",
    "**/*.npz",
    "**/*.npy",
    "**/*prediction*",
    "**/*pred*",
    "**/*external*",
    "**/*threshold*",
    "**/*sweep*",
]

hits = []
for pattern in patterns:
    hits.extend(glob.glob(os.path.join(SEARCH_ROOT, pattern), recursive=True))

rows = []
seen = set()
for p in hits:
    if p in seen or not os.path.isfile(p):
        continue
    seen.add(p)
    name = os.path.basename(p).lower()
    if any(k in name for k in ["pred", "prediction", "external", "threshold", "sweep", ".csv", ".npz", ".npy"]):
        rows.append({
            "path": p,
            "filename": os.path.basename(p),
            "size_mb": os.path.getsize(p) / (1024**2),
            "modified": pd.to_datetime(os.path.getmtime(p), unit="s")
        })

files_df = pd.DataFrame(rows)
if len(files_df):
    files_df = files_df.sort_values("modified", ascending=False)
display(files_df.head(200))

,path,filename,size_mb,modified
17,/content/drive/MyDrive/Colab Notebooks/verify_...,verify_drone_model_figures_thresholds_colab.ipynb,0.027859,2026-05-12 13:59:12
21,/content/drive/MyDrive/drone_audio_processed/f...,fig8_snr_sweep.png,0.261013,2026-05-12 06:50:58
20,/content/drive/MyDrive/drone_audio_processed/f...,cnntf_v2_threshold_sweep.png,0.092432,2026-05-10 19:28:42
12,/content/drive/MyDrive/drone_audio_processed/f...,cnntf_v2_external_cm.png,0.088851,2026-05-10 19:28:41
11,/content/drive/MyDrive/drone_audio_processed/f...,cnntf_external_roc.png,0.077259,2026-05-09 15:41:57
19,/content/drive/MyDrive/drone_audio_processed/f...,cnntf_threshold_sweep.png,0.173738,2026-05-09 15:41:56
10,/content/drive/MyDrive/drone_audio_processed/f...,cnntf_external_cm.png,0.080316,2026-05-09 15:41:55
7,/content/drive/MyDrive/Colab Notebooks/lwcnn_e...,lwcnn_external_eval_fix.ipynb,0.279880,2026-05-09 15:39:11
18,/content/drive/MyDrive/drone_audio_processed/f...,lwcnn_threshold_sweep.png,0.179235,2026-05-09 15:39:10
8,/content/drive/MyDrive/drone_audio_processed/f...,lwcnn_external_roc.png,0.077237,2026-05-09 15:39:10


## 4. Set your prediction file paths

Edit these four paths.

You need one internal and one external prediction file per model:

- LWCNN internal predictions
- LWCNN external predictions
- CNN-Transformer internal predictions
- CNN-Transformer external predictions

The files should contain labels and probabilities. If your external file is window-level, include `clip_id` so it can be aggregated to clip level using max probability.

In [4]:
# === EDIT THESE PATHS ===
LWCNN_INTERNAL_PATH = "/content/drive/MyDrive/path/to/lwcnn_internal_predictions.csv"
LWCNN_EXTERNAL_PATH = "/content/drive/MyDrive/path/to/lwcnn_external_predictions.csv"

CNN_TRANSFORMER_INTERNAL_PATH = "/content/drive/MyDrive/path/to/cnntransformer_internal_predictions.csv"
CNN_TRANSFORMER_EXTERNAL_PATH = "/content/drive/MyDrive/path/to/cnntransformer_external_predictions.csv"

# Set to True if your external prediction CSV is window-level and contains a clip/file identifier.
# If True, the notebook aggregates each clip using max predicted drone probability.
LWCNN_EXTERNAL_IS_WINDOW_LEVEL = False
CNN_TRANSFORMER_EXTERNAL_IS_WINDOW_LEVEL = False

# If external window-level files are used, set the clip-id column name here.
CLIP_ID_COLUMN = "clip_id"

# Thresholds used in your report.
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.75, 0.95]

# Thresholds used for confusion matrix figures.
FIGURE_THRESHOLDS = {
    "LWCNN": [0.50, 0.30],
    "CNN-Transformer": [0.50, 0.05],
}

## 5. Optional: expected values from your report

This cell is used only for checking consistency.  
The notebook will compare the values it calculates against these numbers and flag differences.

Update these if your report values change.

In [5]:
EXPECTED_VALUES = {
    ("LWCNN", 0.50): {
        "int_drone_f1": 0.9815,
        "ext_recall": 0.1000,
        "ext_drone_f1": 0.1818,
    },
    ("LWCNN", 0.30): {
        "int_drone_f1": 0.9659,
        "ext_recall": 0.3333,
        "ext_drone_f1": 0.4545,
    },
    ("CNN-Transformer", 0.50): {
        "int_drone_f1": 0.9836,
        "ext_recall": 0.0667,
        "ext_drone_f1": 0.1143,
    },
    ("CNN-Transformer", 0.05): {
        "int_drone_f1": 0.9177,
        "ext_recall": 0.8000,
        "ext_drone_f1": 0.4800,
    },
}

TOLERANCE = 0.002  # allows small rounding differences

## 6. Helper functions

These functions automatically handle common column names and label formats.

In [6]:
def load_prediction_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    suffix = Path(path).suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".npz":
        data = np.load(path, allow_pickle=True)
        return pd.DataFrame({k: data[k] for k in data.files})
    if suffix == ".npy":
        arr = np.load(path, allow_pickle=True)
        if arr.ndim == 1:
            return pd.DataFrame({"y_prob": arr})
        return pd.DataFrame(arr)

    raise ValueError(f"Unsupported file type: {path}")

def infer_column(df, candidates, required=True, label="column"):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    # Fuzzy contains match
    for c in df.columns:
        cl = c.lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    if required:
        raise ValueError(f"Could not infer {label}. Columns found: {list(df.columns)}")
    return None

def normalise_binary_labels(series):
    s = series.copy()

    # Already numeric
    if pd.api.types.is_numeric_dtype(s):
        values = s.astype(float).values
        # If labels are 0/1, keep them.
        unique = set(np.unique(values[~np.isnan(values)]))
        if unique.issubset({0.0, 1.0}):
            return values.astype(int)

    # String labels
    s_str = s.astype(str).str.strip().str.lower()

    positive_tokens = {"1", "drone", "uav", "positive", "pos", "true", "yes"}
    negative_tokens = {"0", "no drone", "no_drone", "non-drone", "nondrone", "background", "helicopter", "negative", "neg", "false", "no"}

    out = []
    for v in s_str:
        if v in positive_tokens or "drone" == v:
            out.append(1)
        elif v in negative_tokens:
            out.append(0)
        elif "drone" in v and "no" not in v and "non" not in v:
            out.append(1)
        else:
            out.append(0)
    return np.asarray(out).astype(int)

def extract_y_true_y_prob(df):
    label_col = infer_column(
        df,
        ["y_true", "true", "true_label", "label", "labels", "target", "class", "ground_truth"],
        required=True,
        label="true label column"
    )

    prob_col = infer_column(
        df,
        ["y_prob", "prob", "probability", "drone_probability", "pred_prob", "prediction", "score", "p_drone"],
        required=True,
        label="probability column"
    )

    y_true = normalise_binary_labels(df[label_col])
    y_prob = pd.to_numeric(df[prob_col], errors="coerce").values.astype(float)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    return y_true, y_prob, label_col, prob_col

def aggregate_external_to_clip(df, clip_id_col=CLIP_ID_COLUMN):
    if clip_id_col not in df.columns:
        raise ValueError(
            f"clip_id column '{clip_id_col}' not found. Columns found: {list(df.columns)}"
        )

    label_col = infer_column(
        df,
        ["y_true", "true", "true_label", "label", "labels", "target", "class", "ground_truth"],
        required=True,
        label="true label column"
    )
    prob_col = infer_column(
        df,
        ["y_prob", "prob", "probability", "drone_probability", "pred_prob", "prediction", "score", "p_drone"],
        required=True,
        label="probability column"
    )

    tmp = df[[clip_id_col, label_col, prob_col]].copy()
    tmp["_y_true"] = normalise_binary_labels(tmp[label_col])
    tmp["_y_prob"] = pd.to_numeric(tmp[prob_col], errors="coerce")

    # Clip-level probability = max window probability.
    # Clip label = max label, assuming any drone window/clip means drone.
    clip_df = tmp.groupby(clip_id_col, as_index=False).agg(
        y_true=("_y_true", "max"),
        y_prob=("_y_prob", "max")
    )

    return clip_df

def binary_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    labels = [0, 1]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=labels).ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0

    return {
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

def harmonic_mean(a, b):
    return 0.0 if (a + b) == 0 else 2 * a * b / (a + b)

def make_threshold_sweep(model_name, y_int, p_int, y_ext, p_ext, thresholds):
    rows = []
    for t in thresholds:
        int_m = binary_metrics(y_int, p_int, t)
        ext_m = binary_metrics(y_ext, p_ext, t)
        rows.append({
            "model": model_name,
            "threshold": t,
            "int_drone_f1": int_m["f1"],
            "ext_recall": ext_m["recall"],
            "ext_drone_f1": ext_m["f1"],
            "combined_hm_int_f1_ext_recall": harmonic_mean(int_m["f1"], ext_m["recall"]),
            "external_hm_recall_f1": harmonic_mean(ext_m["recall"], ext_m["f1"]),
            "int_tn": int_m["tn"],
            "int_fp": int_m["fp"],
            "int_fn": int_m["fn"],
            "int_tp": int_m["tp"],
            "ext_tn": ext_m["tn"],
            "ext_fp": ext_m["fp"],
            "ext_fn": ext_m["fn"],
            "ext_tp": ext_m["tp"],
        })
    return pd.DataFrame(rows)

def check_monotonic_recall(sweep_df):
    rows = []
    for model, g in sweep_df.groupby("model"):
        g = g.sort_values("threshold")
        recalls = g["ext_recall"].values
        thresholds = g["threshold"].values
        for i in range(1, len(recalls)):
            if recalls[i] > recalls[i-1] + 1e-12:
                rows.append({
                    "model": model,
                    "problem": "external recall increased as threshold increased",
                    "threshold_previous": thresholds[i-1],
                    "recall_previous": recalls[i-1],
                    "threshold_current": thresholds[i],
                    "recall_current": recalls[i],
                })
    return pd.DataFrame(rows)

def plot_confusion(ax, y_true, y_prob, threshold, title):
    m = binary_metrics(y_true, y_prob, threshold)
    matrix = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])

    im = ax.imshow(matrix)
    ax.set_title(title)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No Drone", "Drone"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["No Drone", "Drone"])
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    row_sums = matrix.sum(axis=1, keepdims=True)
    pct = np.divide(matrix, row_sums, out=np.zeros_like(matrix, dtype=float), where=row_sums != 0) * 100

    labels = [["TN", "FP"], ["FN", "TP"]]
    for i in range(2):
        for j in range(2):
            ax.text(
                j, i,
                f"{labels[i][j]}\n{matrix[i, j]:,}\n({pct[i, j]:.1f}%)",
                ha="center", va="center"
            )

    return im

def plot_threshold_sweep(model_name, sweep_df, optimal_threshold, out_path):
    g = sweep_df[sweep_df["model"] == model_name].sort_values("threshold")

    plt.figure(figsize=(7, 5))
    plt.plot(g["threshold"], g["int_drone_f1"], marker="o", label="Internal Drone F1")
    plt.plot(g["threshold"], g["ext_recall"], marker="s", label="External Drone Recall")
    plt.plot(g["threshold"], g["ext_drone_f1"], marker="^", label="External Drone F1")
    plt.plot(g["threshold"], g["combined_hm_int_f1_ext_recall"], marker="D", label="Combined (HM)")
    plt.axvline(optimal_threshold, linestyle=":", label=f"Selected={optimal_threshold:g}")
    plt.title(f"{model_name}: Threshold vs Metrics")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.ylim(-0.05, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()
    print("Saved:", out_path)

## 7. Load predictions

This cell loads your files and prints the inferred columns.

In [9]:
def load_model_predictions(model_name, internal_path, external_path, external_is_window_level):
    int_df = load_prediction_file(internal_path)
    ext_df = load_prediction_file(external_path)

    print(f"\n{model_name} internal file columns:", list(int_df.columns))
    y_int, p_int, int_label_col, int_prob_col = extract_y_true_y_prob(int_df)
    print(f"{model_name} internal: using label column '{int_label_col}', probability column '{int_prob_col}'")
    print(f"{model_name} internal samples:", len(y_int), "positive:", int(y_int.sum()), "negative:", int((1-y_int).sum()))

    print(f"\n{model_name} external file columns:", list(ext_df.columns))
    if external_is_window_level:
        ext_df = aggregate_external_to_clip(ext_df, CLIP_ID_COLUMN)
        print(f"{model_name} external aggregated to clip level. New columns:", list(ext_df.columns))

    y_ext, p_ext, ext_label_col, ext_prob_col = extract_y_true_y_prob(ext_df)
    print(f"{model_name} external: using label column '{ext_label_col}', probability column '{ext_prob_col}'")
    print(f"{model_name} external samples:", len(y_ext), "positive:", int(y_ext.sum()), "negative:", int((1-y_ext).sum()))

    return {
        "y_int": y_int,
        "p_int": p_int,
        "y_ext": y_ext,
        "p_ext": p_ext,
    }

preds = {
    "LWCNN": load_model_predictions(
        "LWCNN",
        LWCNN_INTERNAL_PATH,
        LWCNN_EXTERNAL_PATH,
        LWCNN_EXTERNAL_IS_WINDOW_LEVEL,
    ),
    "CNN-Transformer": load_model_predictions(
        "CNN-Transformer",
        CNN_TRANSFORMER_INTERNAL_PATH,
        CNN_TRANSFORMER_EXTERNAL_PATH,
        CNN_TRANSFORMER_EXTERNAL_IS_WINDOW_LEVEL,
    ),
}

FileNotFoundError: File not found: /content/drive/MyDrive/path/to/lwcnn_internal_predictions.csv

## 8. Generate the threshold sweep table

This is the table you should use in the report.

In [ ]:
sweeps = []
for model_name, d in preds.items():
    sweeps.append(
        make_threshold_sweep(
            model_name=model_name,
            y_int=d["y_int"],
            p_int=d["p_int"],
            y_ext=d["y_ext"],
            p_ext=d["p_ext"],
            thresholds=THRESHOLDS,
        )
    )

sweep_df = pd.concat(sweeps, ignore_index=True)

report_table = sweep_df[[
    "model",
    "threshold",
    "int_drone_f1",
    "ext_recall",
    "ext_drone_f1",
    "combined_hm_int_f1_ext_recall",
    "external_hm_recall_f1",
    "int_tn", "int_fp", "int_fn", "int_tp",
    "ext_tn", "ext_fp", "ext_fn", "ext_tp",
]].copy()

# Rounded display version
display_df = report_table.copy()
for col in ["int_drone_f1", "ext_recall", "ext_drone_f1", "combined_hm_int_f1_ext_recall", "external_hm_recall_f1"]:
    display_df[col] = display_df[col].round(4)

display(display_df)

## 9. Check for impossible or inconsistent trends

External recall should not increase as the threshold increases if it is calculated from one fixed set of probabilities.  
If this check flags a row, your table/figures are probably being mixed across runs.

In [ ]:
monotonic_problems = check_monotonic_recall(sweep_df)
if len(monotonic_problems) == 0:
    print("PASS: External recall is monotonic with threshold for both models.")
else:
    print("WARNING: External recall inconsistency detected.")
    display(monotonic_problems)

## 10. Compare calculated values against expected report values

If anything fails here, your report value does not match the prediction files used by this notebook.

In [ ]:
comparison_rows = []
for (model_name, threshold), expected in EXPECTED_VALUES.items():
    row = sweep_df[(sweep_df["model"] == model_name) & (np.isclose(sweep_df["threshold"], threshold))]
    if row.empty:
        comparison_rows.append({
            "model": model_name,
            "threshold": threshold,
            "metric": "ALL",
            "expected": "row missing",
            "calculated": np.nan,
            "difference": np.nan,
            "status": "FAIL"
        })
        continue

    row = row.iloc[0]
    for metric, expected_value in expected.items():
        calc = float(row[metric])
        diff = calc - expected_value
        comparison_rows.append({
            "model": model_name,
            "threshold": threshold,
            "metric": metric,
            "expected": expected_value,
            "calculated": calc,
            "difference": diff,
            "status": "PASS" if abs(diff) <= TOLERANCE else "FAIL"
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

if (comparison_df["status"] == "FAIL").any():
    print("Some values do not match. Use the calculated values from this notebook, or check that the prediction files are from the correct run.")
else:
    print("PASS: Expected report values match the loaded predictions within tolerance.")

## 11. Generate confusion matrix figures

This creates the figures for the selected thresholds and saves them to `/content/verified_figures/`.

In [ ]:
out_dir = Path("/content/verified_figures")
out_dir.mkdir(exist_ok=True)

for model_name, thresholds in FIGURE_THRESHOLDS.items():
    d = preds[model_name]

    fig, axes = plt.subplots(1, len(thresholds), figsize=(6 * len(thresholds), 5))
    if len(thresholds) == 1:
        axes = [axes]
    for ax, t in zip(axes, thresholds):
        plot_confusion(ax, d["y_int"], d["p_int"], t, f"{model_name} Internal Test Set\n(threshold = {t:g})")
    fig.tight_layout()
    internal_path = out_dir / f"{model_name.replace(' ', '_').replace('-', '_')}_internal_confusion.png"
    fig.savefig(internal_path, dpi=200)
    plt.show()
    print("Saved:", internal_path)

    fig, axes = plt.subplots(1, len(thresholds), figsize=(6 * len(thresholds), 5))
    if len(thresholds) == 1:
        axes = [axes]
    for ax, t in zip(axes, thresholds):
        plot_confusion(ax, d["y_ext"], d["p_ext"], t, f"{model_name} External Test Set\n(threshold = {t:g})")
    fig.tight_layout()
    external_path = out_dir / f"{model_name.replace(' ', '_').replace('-', '_')}_external_confusion.png"
    fig.savefig(external_path, dpi=200)
    plt.show()
    print("Saved:", external_path)

## 12. Generate threshold-metric figures

This creates the threshold sweep plots from the same predictions used for the tables and confusion matrices.

In [ ]:
# Select thresholds based on your current report:
OPTIMAL_THRESHOLDS = {
    "LWCNN": 0.30,
    "CNN-Transformer": 0.05,
}

for model_name, t_opt in OPTIMAL_THRESHOLDS.items():
    out_path = Path("/content/verified_figures") / f"{model_name.replace(' ', '_').replace('-', '_')}_threshold_sweep.png"
    plot_threshold_sweep(model_name, sweep_df, t_opt, out_path)

## 13. Save all verified outputs

The CSV files and images can be downloaded from Colab's left file panel.

In [ ]:
tables_dir = Path("/content/verified_tables")
tables_dir.mkdir(exist_ok=True)

full_csv = tables_dir / "verified_full_threshold_sweep.csv"
report_csv = tables_dir / "verified_report_threshold_table.csv"
comparison_csv = tables_dir / "report_value_consistency_check.csv"

sweep_df.to_csv(full_csv, index=False)
display_df.to_csv(report_csv, index=False)
comparison_df.to_csv(comparison_csv, index=False)

print("Saved:", full_csv)
print("Saved:", report_csv)
print("Saved:", comparison_csv)
print("Saved figures in:", out_dir)

## 14. Report-ready tables

These are compact versions of the threshold tables. Copy these into your report if the consistency checks pass.

In [ ]:
for model_name in ["LWCNN", "CNN-Transformer"]:
    print("\n" + "="*80)
    print(model_name)
    print("="*80)
    compact = display_df[display_df["model"] == model_name][[
        "threshold", "int_drone_f1", "ext_recall", "ext_drone_f1", "combined_hm_int_f1_ext_recall"
    ]].copy()
    display(compact)

## 15. Confusion matrix sanity check only

Use this section when you only have confusion matrix counts and want to calculate precision, recall, and F1.

This does not generate a full threshold sweep because a full sweep needs predicted probabilities.

In [ ]:
def metrics_from_counts(tn, fp, fn, tp):
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
    return {
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

manual_checks = [
    {"model": "LWCNN internal", "threshold": 0.50, **metrics_from_counts(15558, 255, 368, 16513)},
    {"model": "LWCNN internal", "threshold": 0.30, **metrics_from_counts(14900, 913, 261, 16620)},
    {"model": "LWCNN external", "threshold": 0.50, **metrics_from_counts(60, 0, 27, 3)},
    {"model": "LWCNN external", "threshold": 0.30, **metrics_from_counts(56, 4, 20, 10)},
    {"model": "CNN-Transformer internal", "threshold": 0.50, **metrics_from_counts(15629, 184, 368, 16513)},
    {"model": "CNN-Transformer internal", "threshold": 0.05, **metrics_from_counts(12930, 2883, 123, 16758)},
]

manual_df = pd.DataFrame(manual_checks)
display(manual_df.round(4))